# Stage 1 — which signal-processing method discriminates tremor best?

Compares **12 methods** on equal footing. Each reduces a recording to a power
spectrum, from which the same interpretable descriptors are computed —
**max frequency, mean frequency**, median frequency, spread, entropy, Q-factor,
peak share, IQR, low/high ratio, total power.

| method | family |
|---|---|
| `welch` | non-parametric PSD (reference) |
| `stft256`, `stft512` | short-time Fourier, two windows |
| `multitaper` | Slepian multitaper PSD |
| `cwt` | continuous wavelet (Morlet) |
| `hht`, `hht_imf2plus` | Hilbert-Huang, all IMFs / IMF1 dropped |
| `sst` | synchrosqueezed transform |
| `wavelet_packet` | wavelet packet band energies |
| `stransform` | Stockwell S-transform |
| `vmd` | variational mode decomposition |
| `ar16` | parametric autoregressive spectrum |

**Two axes reported separately** — N-vs-Tremor and PD-vs-ET. They behave very
differently and a single accuracy hides that. PD-vs-ET uses **balanced
accuracy**: the majority baseline is 0.833, so raw accuracy is misleading.

**Two levels of evidence**, both required before calling a method best:
univariate with **BH-corrected q-values** (the grid is ~120 tests), and
multivariate LOSO with a **paired bootstrap CI** against the reference method.

## 1. Setup

In [ ]:
import sys, os
if os.path.basename(os.getcwd()) == "tfbench": os.chdir("..")
sys.path.insert(0, os.getcwd())
import numpy as np, matplotlib.pyplot as plt
from tremor.quaternion_data import load_quaternion_recordings
from tfbench.transforms import METHODS, F_MIN, F_MAX
from tfbench.descriptors import DESCRIPTOR_NAMES, describe
from tfbench import benchmark as B

DATA_ROOT, ACTION = "Data", "OUT"
recs = load_quaternion_recordings(DATA_ROOT, action=ACTION, mode="angular_velocity")
print(len(recs), "recordings,", len({r.subject for r in recs}), "patients")
print("methods:", list(METHODS))

## 2. Sanity check — does every method find a known frequency?

A method that cannot recover a synthetic 6 Hz tone cannot be trusted on real
tremor. Run this first; it catches frequency-axis mapping bugs immediately.

In [ ]:
t = np.arange(1500)/100.0
np.random.seed(0)
synth = np.stack([np.sin(2*np.pi*6*t) + 0.3*np.random.randn(1500) for _ in range(3)])
print("true peak = 6.00 Hz, noise sd 0.3")
for name, fn in METHODS.items():
    try:
        f, P = fn(synth)
        pk = f[np.argmax(P)]
        print(f"  {name:>15} {len(f):>4} bins   peak {pk:6.2f} Hz  "
              f"{'ok' if abs(pk-6) < 0.8 else '<-- OFF'}")
    except Exception as e:
        print(f"  {name:>15} FAILED {type(e).__name__}: {e}")

### 2b. What the spectra actually look like

Plot one recording per class through a few methods. Eyeball this before
trusting any number below.

In [ ]:
show = ["welch", "stft256", "cwt", "hht_imf2plus", "multitaper", "ar16"]
pick = {}
for r in recs: pick.setdefault(r.y, r)
fig, axes = plt.subplots(len(show), 1, figsize=(9, 2.1*len(show)), sharex=True)
for ax, meth in zip(np.atleast_1d(axes), show):
    for cls, r in sorted(pick.items()):
        f, P = METHODS[meth](r.x)
        ax.plot(f, P/ (P.max()+1e-20), label=["N","PD","ET"][cls], lw=1.4)
    ax.set_ylabel(meth, fontsize=8); ax.grid(alpha=.3)
np.atleast_1d(axes)[0].legend(fontsize=8, ncol=3)
np.atleast_1d(axes)[-1].set_xlabel("frequency (Hz)")
plt.tight_layout(); plt.show()

## 3. Compute descriptors for every method

One pass; the tables are reused by everything below. HHT and the S-transform
are the slow ones (a few minutes total).

In [ ]:
tables = B.build_all(recs, fs=100.0)
print("\nmethods with usable tables:", list(tables))

## 4. Univariate screen — which single descriptor separates best?

This is the "max frequency / mean frequency" question asked directly.
**Read the BH q column, not the raw p.** With ~120 tests, several raw p<0.05
are expected by chance alone.

In [ ]:
rows_nt = B.screen(tables, axis="N_vs_Tremor", top=15)
rows_pe = B.screen(tables, axis="PD_vs_ET",   top=15)

## 5. Method ranking — full descriptor set, patient-level LOSO

The `*` marks methods whose paired CI against the reference **excludes zero**.
Anything without a `*` is not distinguishable from `welch`, however good its
point estimate looks.

In [ ]:
rank_nt = B.rank_methods(tables, axis="N_vs_Tremor", reference="welch")
rank_pe = B.rank_methods(tables, axis="PD_vs_ET",   reference="welch")

## 6. Summary chart + pick the top methods for Stage 2

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for ax, rank, title, base in [(axes[0], rank_nt, "N vs Tremor", 0.5),
                              (axes[1], rank_pe, "PD vs ET", 0.5)]:
    names = [r[0] for r in rank]; vals = [r[1] for r in rank]
    ax.barh(range(len(names)), vals, color="steelblue")
    ax.axvline(base, color="crimson", ls="--", lw=1, label="chance (balanced)")
    ax.set_yticks(range(len(names))); ax.set_yticklabels(names, fontsize=8)
    ax.set_xlabel("balanced accuracy"); ax.set_title(title); ax.legend(fontsize=8)
    ax.invert_yaxis()
plt.tight_layout(); plt.show()

TOP_METHODS = [r[0] for r in rank_pe[:4]]
print("top 4 on PD-vs-ET, carry these to Stage 2:", TOP_METHODS)
import json; json.dump(TOP_METHODS, open("artifacts/tfbench_top_methods.json","w"))

## 7. How to read this

* A high point estimate with a CI spanning zero is **not** a better method.
* A descriptor surviving BH q<0.05 is a real univariate effect; a raw p<0.05
  in a 120-test grid usually is not.
* If nothing beats `welch`, that is a result: the tremor information is in the
  spectrum, not in the choice of estimator, and Stage 2 should test whether a
  deep model extracts more from the same representation.